In [1]:
import sys 
import os 

sys.path.append(os.path.join('..', '..'))

In [2]:
from pathlib import Path
import pandas as pd 
import zipfile
import numpy as np
import re 
import unicodedata

from config.bairro_cleaning_config import MAPA_BAIRRO, BAIRROS_OFICIAIS

In [3]:
data_dir = Path("../../data/ibge2022")
raw_dir = data_dir / "raw"

Vamos utilizar quatro arquivos necessários para a extração dos dados, já disponibilizados pelo IBGE em nível de bairro:

- `Agregados_por_bairro_basico_BR.zip`: contém os dados básicos dos bairros, incluindo códigos e nomes das unidades territoriais, município de pertencimento e população total. Esse arquivo foi utilizado para identificar os bairros de Curitiba e obter a população total de cada bairo;
- `Agregados_por_bairro_alfabetizacao_BR.zip`: contém os dados de alfabetização da população residente, organizados por grupos etários. Esse arquivo foi utilizado para calcular a quantidade de pessoas alfabetizadas de 15 anos ou mais e a taxa de alfabetização por bairro;
- `Agregados_por_bairro_rendimento_responsavel_BR.zip`: contém os dados de rendimento das pessoas responsáveis pelos domicílios particulares ocupados. Esse arquivo foi utilizado para obter o número de responsáveis, o número de moradores em domicílios particulares ocupados e o rendimento nominal médio mensal dos responsáveis por bairro;
- `Agregados_por_bairros_caracteristicas_domicilio2_BR_20250417.zip` e `Agregados_por_bairros_caracteristicas_domicilio1_BR.zip`: será utilizado para obter indicadores de qualidade de vida da população, como: a presença de banheiros sanitários nas moradias particulares, sistema de esgoto precário, ausência de rede geral de água, presença de lixo destinado de forma inadequada e domicílios improvisados.
- `dicionario_de_dados_renda_responsavel_.xlsx`: contém o dicionário de dados do arquivo de rendimento do responsável.

Os dados podem ser obtidos em:

https://www.ibge.gov.br/estatisticas/downloads-estatisticas.html

In [4]:
for zip_path in raw_dir.glob("*.zip"):
    pasta_saida = raw_dir / zip_path.stem 
    pasta_saida.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(pasta_saida)
    
    print(f"Extraído: {zip_path.name}")

Extraído: Agregados_por_bairros_caracteristicas_domicilio1_BR.zip


In [5]:
zip_path.unlink()

Agora utilizaremos o arquivo `Agregados_por_bairros_basico_BR.csv`. Utilizaremos esse arquivo para identificar os bairros e a população total de cada um deles, filtrando pelo código municial:

`CD_MUN = 4106902`

https://www.ibge.gov.br/cidades-e-estados/pr/curitiba.html

In [6]:
arquivo_basico = raw_dir / "Agregados_por_bairros_basico_BR_20250417/Agregados_por_bairros_basico_BR.csv"
arquivo_pessoa_renda = raw_dir / "Agregados_por_bairros_renda_responsavel_BR_csv/Agregados_por_bairros_renda_responsavel_BR.csv"
arquivo_alfabetizacao = raw_dir / "Agregados_por_bairros_alfabetizacao_BR/Agregados_por_bairros_alfabetizacao_BR.csv"
arquivo_domicilio1 = raw_dir / "Agregados_por_bairros_caracteristicas_domicilio1_BR/Agregados_por_bairros_caracteristicas_domicilio1_BR.csv"
arquivo_domicilio2 = raw_dir / "Agregados_por_bairros_caracteristicas_domicilio2_BR_20250417/Agregados_por_bairros_caracteristicas_domicilio2_BR.csv"

In [7]:
df = pd.read_csv(arquivo_basico, sep=";", encoding="latin1", dtype=str)

In [8]:
renda = pd.read_csv(arquivo_pessoa_renda, sep=";", encoding="latin1", dtype=str)

In [9]:
alfabetizacao = pd.read_csv(arquivo_alfabetizacao, sep=";", encoding="latin1", dtype=str)

In [10]:
domicilio1 = pd.read_csv(arquivo_domicilio1, sep=";", encoding="latin1", dtype=str)

In [11]:
domicilio2 = pd.read_csv(arquivo_domicilio2, sep=";", encoding="latin1", dtype=str)

In [12]:
df.columns

Index(['CD_BAIRRO', 'NM_BAIRRO', 'CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF',
       'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST',
       'CD_NU', 'NM_NU', 'CD_AGLOM', 'NM_AGLOM', 'CD_RGINT', 'NM_RGINT',
       'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'AREA_KM2', 'v0001',
       'v0002', 'v0003', 'v0004', 'v0005', 'v0006', 'v0007'],
      dtype='str')

Vamos executar o filtro de bairro por `CD_MUN`:

In [13]:
df_curitiba = df.loc[df['CD_MUN'] == '4106902']

In [14]:
df_curitiba['CD_MUN'].unique()

<StringArray>
['4106902']
Length: 1, dtype: str

In [15]:
codigo_bairro = df_curitiba[["NM_BAIRRO", "CD_BAIRRO", "NM_MUN", "CD_MUN"]]

In [16]:
codigo_bairro.groupby(["CD_BAIRRO"], dropna=True).first()

,NM_BAIRRO,NM_MUN,CD_MUN
CD_BAIRRO,,,
4106902001,Centro,Curitiba,4106902
4106902002,São Francisco,Curitiba,4106902
4106902003,Centro Cívico,Curitiba,4106902
4106902004,Alto da Glória,Curitiba,4106902
4106902005,Alto da Rua XV,Curitiba,4106902
...,...,...,...
4106902071,Campo de Santana,Curitiba,4106902
4106902072,Ganchinho,Curitiba,4106902
4106902073,Umbará,Curitiba,4106902


O código dos bairros continua o mesmo do censo de 2010, bem como a quantidade de bairros oficiais.

Vamos pegar a coluna `CD_BAIRRO` e remover espaços em branco no começo e no fim e trasformar numa coleção de códigos únicos de bairros de Curitiba:

In [17]:
bairros_curitiba = set(
    df_curitiba["CD_BAIRRO"].astype(str).str.strip()
)

In [18]:
len(bairros_curitiba)

75

Assim, conseguimos utilizar a coleção `bairros_curitiba` como filtro para o dataset de alfabetização e renda:

In [19]:
curitiba_alfabetizacao = alfabetizacao[
    alfabetizacao["CD_BAIRRO"].astype(str).str.strip().isin(bairros_curitiba)
].copy()

curitiba_alfabetizacao.head()

,CD_BAIRRO,NM_BAIRRO,V00644,V00645,V00646,V00647,V00648,V00649,V00650,V00651,...,V00996,V00997,V00998,V00999,V01000,V01001,V01002,V01003,V01004,V01005
11904,4106902001,Centro,1772,5003,5871,4408,3201,2406,2016,1858,...,167,717,40,20,308,510,205,202,965,774
11905,4106902002,São Francisco,169,332,609,545,426,348,302,324,...,21,103,9,12,43,70,49,44,93,99
11906,4106902003,Centro Cívico,160,328,431,426,364,323,241,256,...,16,80,8,6,16,32,20,32,16,14
11907,4106902004,Alto da Glória,239,422,576,512,509,510,390,377,...,20,113,13,16,44,48,26,33,44,44
11908,4106902005,Alto da Rua XV,333,496,673,691,688,638,505,505,...,53,181,26,10,53,72,30,39,81,84


In [20]:
curitiba_pessoa_renda = renda[
    renda["CD_BAIRRO"].astype(str).str.strip().isin(bairros_curitiba)
].copy()

curitiba_pessoa_renda.head()

,CD_BAIRRO,NM_BAIRRO,V06001,V06002,V06003,V06004,V06005
11737,4106902001,Centro,22215,38081,"0,92","6595,93","185958547,73"
11738,4106902002,São Francisco,2394,4914,"1,27","7030,76","47004965,59"
11739,4106902003,Centro Cívico,2348,4453,"1,04","10278,65","4086259628,19"
11740,4106902004,Alto da Glória,3159,6242,"1,02","10307,91","90064694,87"
11741,4106902005,Alto da Rua XV,3858,8016,"1,12","9020,56","73867666,85"


# 1. Extraindo informações de população por bairro

Utilizando o arquivo `Agregados_por_bairros_basico_BR.csv` podemos extrair a informação da população por bairro filtrando os bairros de curitiba e extraindo as informações da coluna `v001`:

In [21]:
coluna_populacao = "v0001"

In [22]:
df_curitiba[coluna_populacao] = pd.to_numeric(
    df[coluna_populacao],
    errors="coerce",
)

In [23]:
populacao_por_bairro = (
    df_curitiba
    .groupby(["CD_BAIRRO", "NM_BAIRRO"])[coluna_populacao]
    .sum(min_count=1)
    .reset_index(name="populacao_2022")
)

In [24]:
populacao_por_bairro["populacao_2022"] = (
    populacao_por_bairro["populacao_2022"].astype("Int64")
)

In [25]:
pop_total_bairros = populacao_por_bairro["populacao_2022"].sum()

pop_total_bairros

np.int64(1773718)

In [26]:
populacao_por_bairro.head(3)

,CD_BAIRRO,NM_BAIRRO,populacao_2022
0,4106902001,Centro,38671
1,4106902002,São Francisco,5008
2,4106902003,Centro Cívico,4473


# 2. Extraindo informação de renda 

Para extrair informações relcionadas à renda das pessoas, utirelizamores o arquivo `Agregados_por_bairro_rendimento_responsavel_BR`. 

Esse arquivo será utilizado para obter indicadores de renda em nível de bairro, especialmente o rendimento nominal médio mensal dos responsáveis pelos domicílios. 

As variáveis que serão extraídas são:

- `V06001` = pessoas responsáveis por domicílios particulares ocupados;
- `V06002` = moradores em domicílios particulares ocupados;
- `V06003` = variância do número de moradores;
- `V06004` = rendimento nominal médio mensal das pessoas responsáveis com rendimento;
- `V06005` = variância do rendimento nominal mensal.

In [27]:
renda.columns

Index(['CD_BAIRRO', 'NM_BAIRRO', 'V06001', 'V06002', 'V06003', 'V06004',
       'V06005'],
      dtype='str')

In [28]:
curitiba_pessoa_renda.head(3)

,CD_BAIRRO,NM_BAIRRO,V06001,V06002,V06003,V06004,V06005
11737,4106902001,Centro,22215,38081,"0,92","6595,93","185958547,73"
11738,4106902002,São Francisco,2394,4914,"1,27","7030,76","47004965,59"
11739,4106902003,Centro Cívico,2348,4453,"1,04","10278,65","4086259628,19"


In [29]:
curitiba_pessoa_renda = curitiba_pessoa_renda.rename(columns={
    "V06001": "resp_domicilios_particulares",
    "V06002": "moradores_domicilios_particulares",
    "V06004": "rendimento_medio_responsavel",
})

In [30]:
curitiba_pessoa_renda["rendimento_medio_responsavel"] = (
    curitiba_pessoa_renda["rendimento_medio_responsavel"]
    .astype("string")
    .str.strip()
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)  
    .pipe(pd.to_numeric, errors="coerce")
)

In [31]:
salario_minimo_2022 = 1212

In [32]:
curitiba_pessoa_renda["rendimento_medio_responsavel"] = pd.to_numeric(
    curitiba_pessoa_renda["rendimento_medio_responsavel"],
    errors="coerce"
)

In [33]:
curitiba_pessoa_renda["rendimento_medio_responsavel_sm"] = (
    curitiba_pessoa_renda["rendimento_medio_responsavel"] / salario_minimo_2022
)

In [34]:
curitiba_pessoa_renda = curitiba_pessoa_renda.drop(
    columns=["V06003", "V06005"],
    errors="ignore"
)

In [35]:
curitiba_pessoa_renda.head(3)

,CD_BAIRRO,NM_BAIRRO,resp_domicilios_particulares,moradores_domicilios_particulares,rendimento_medio_responsavel,rendimento_medio_responsavel_sm
11737,4106902001,Centro,22215,38081,6595.93,5.442186
11738,4106902002,São Francisco,2394,4914,7030.76,5.800957
11739,4106902003,Centro Cívico,2348,4453,10278.65,8.480734


# 3. Extraindo informação de alfabetização

Para calcular as taxas de alfabetização dos bairros de Curitiba no censo de 2022, utilizaremos o arquivo:

- `Agregados_por_bairro_alfabetizacao_BR.zip`: utilizado para obter, em nível de bairro, a população por faixa etária e a quantidade de pessoas alfabetizadas por faixa etaária.

Diferentemente do censo de 2010, não será necessário relacionar os setores censitários aos bairros, pois o arquivo já está agregado. O arquivo contém várias colunas refentes à população total por grupos de idade e variaǘeis referentes à população alfabetizada nesses mesmos grupos etários. Dessa forma, é possível calcular a taxa de alfabetização comparando a quantidade de pessoas alfabetiadas com o total de pessoas da mesma faixa etária.

A taxa de alfabetização será calculada para a população de 15 anos ou mais.

Somaremos as seguintes variáveis para obter `pop_15mais`:

- `V00644`: pessoas de 15 a 19 anos;
- `V00645`: pessoas de 20 a 24 anos;
  ...
- `V00656`: pessoas de 75 anos ou mais.

Somares as seguintes variáveis para obter `alfabetizados_15mais`:

- `V00748`: pessoas alfabetizadas de 15 a 19 anos;
- `V00749`: pessoas alfabetizadas de 20 a 24 anos;
...
- `V00760`: pessoas alfabetizadas de 75 anos ou mais.



In [36]:
curitiba_alfabetizacao.head()

,CD_BAIRRO,NM_BAIRRO,V00644,V00645,V00646,V00647,V00648,V00649,V00650,V00651,...,V00996,V00997,V00998,V00999,V01000,V01001,V01002,V01003,V01004,V01005
11904,4106902001,Centro,1772,5003,5871,4408,3201,2406,2016,1858,...,167,717,40,20,308,510,205,202,965,774
11905,4106902002,São Francisco,169,332,609,545,426,348,302,324,...,21,103,9,12,43,70,49,44,93,99
11906,4106902003,Centro Cívico,160,328,431,426,364,323,241,256,...,16,80,8,6,16,32,20,32,16,14
11907,4106902004,Alto da Glória,239,422,576,512,509,510,390,377,...,20,113,13,16,44,48,26,33,44,44
11908,4106902005,Alto da Rua XV,333,496,673,691,688,638,505,505,...,53,181,26,10,53,72,30,39,81,84


In [37]:
col_pop_15mais = []

for i in range(644, 657):
    col_pop_15mais.append(f"V{i:05d}")

In [38]:
col_alfabetizados_15_mais = []

for i in range(748, 761):
    col_alfabetizados_15_mais.append(f"V{i:05d}")

In [39]:
curitiba_alfabetizacao["pop_15mais"] = (
    curitiba_alfabetizacao[col_pop_15mais]
    .astype("string")
    .replace(".", "", regex=False)
    .replace(",", ".", regex=False)
    .apply(pd.to_numeric, errors="coerce")
    .sum(axis=1)
)

In [40]:
curitiba_alfabetizacao["alfabetizados_15mais"] = (
    curitiba_alfabetizacao[col_alfabetizados_15_mais]
    .astype("string")
    .replace(".", "", regex=False)
    .replace(",", ".", regex=False)
    .apply(pd.to_numeric, errors="coerce")
    .sum(axis=1)
)

In [41]:
curitiba_alfabetizacao["pct_alfabetizacao_15mais"] = (
    curitiba_alfabetizacao["alfabetizados_15mais"] / 
    curitiba_alfabetizacao["pop_15mais"] * 100
)

In [42]:
curitiba_alfabetizacao["analfabetos_15_anos_ou_mais"] = (
    curitiba_alfabetizacao["pop_15mais"]
    - curitiba_alfabetizacao["alfabetizados_15mais"]
)

In [43]:
curitiba_alfabetizacao["pct_analfabetismo_15mais"] = (
    curitiba_alfabetizacao["analfabetos_15_anos_ou_mais"] / 
    curitiba_alfabetizacao["pop_15mais"] * 100
)

In [44]:
curitiba_alfabetizacao = curitiba_alfabetizacao[
    [
        "CD_BAIRRO",
        "NM_BAIRRO",
        "pop_15mais",
        "alfabetizados_15mais",
        "pct_alfabetizacao_15mais",
        "pct_analfabetismo_15mais"
    ]
].copy()

In [45]:
curitiba_alfabetizacao.head(3)

,CD_BAIRRO,NM_BAIRRO,pop_15mais,alfabetizados_15mais,pct_alfabetizacao_15mais,pct_analfabetismo_15mais
11904,4106902001,Centro,36334,36121,99.413772,0.586228
11905,4106902002,São Francisco,4550,4533,99.626374,0.373626
11906,4106902003,Centro Cívico,4102,4093,99.780595,0.219405


## 4. Extraindo informações quanto ao saneamento básico

Para calcular informações relacionadas ao saneamento básico dos bairro de Curitiba, utilizaremos os arquivos:

- `Agregados_por_bairros_caracteristicas_domicilio1_BR.csv`
- `Agregados_por_bairros_caracteristicas_domicilio2_BR.csv`

Nessas tabelas, utilizaremos as variáveis:

- `V00001`: domicílios particulares permanentes ocupados;
- `V00002`: domicílios particulares improvisados Ocupados;
- `V00052`: domicílios particulares permanentes ocupados, tipo de espécie é estrutura residencial permanente degradada ou inacabada;
- `V00238`: domicílios particulares permanentes ocupados, não tinham banheiro nem sanitário;
- `V00312`: domicílios particulares permanentes ocupados, destinação do esgoto do banheiro ou sanitário ou buraco para dejeções é fossa rudimentar ou buraco;
- `V00313`: domicílios particulares permanentes ocupados, destinação do esgoto do banheiro ou sanitário ou buraco para dejeções é vala;
- `V00314`: domicílios particulares permanentes ocupados, destinação do esgoto do banheiro ou sanitário ou buraco para dejeções é rio, lago, córrego ou mar;
- `V00315`: domicílios particulares permanentes ocupados, destinação do esgoto do banheiro ou sanitário ou buraco para dejeções é outra forma
- `V00316`: domicílios particulares permanentes ocupados, destinação do esgoto inexistente, pois não tinham banheiro nem sanitário;
- `V00399`: domicílios particulares permanentes ocupados, lixo queimado na propriedade;
- `V00400`: domicílios particulares permanentes ocupados, lixo enterrado na propriedade;
- `V00401`: domicílios particulares permanentes ocupados, lixo jogado em terreno baldio, encosta ou área pública;
- `V00402`: domicílios particulares permanentes ocupados, outro destino do lixo;
- `V00464`: domicílios particulares permanentes ocupados, domicílio não possui ligação à rede geral de distribuição de água.


In [46]:
curitiba_domicilio1 = domicilio1[
    domicilio1["CD_BAIRRO"].astype(str).str.strip().isin(bairros_curitiba)
].copy()

In [47]:
variaveis_domicilio1 = [
    "V00001",
    "V00002",
    "V00052"
]

In [48]:
curitiba_domicilio2 = domicilio2[
    domicilio2["CD_BAIRRO"].astype(str).str.strip().isin(bairros_curitiba)
].copy()

In [49]:
variaveis_domicilio2 = [
    "V00238",
    "V00312",
    "V00313",
    "V00314",
    "V00316",
    "V00464",
    "V00399",
    "V00400",
    "V00401",
    "V00402"
]

In [50]:
for coluna in variaveis_domicilio1:
    curitiba_domicilio1[coluna] = (
        curitiba_domicilio1[coluna]
        .astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [51]:
for coluna in variaveis_domicilio2:
    curitiba_domicilio2[coluna] = (
        curitiba_domicilio2[coluna]
        .astype("string")
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )

In [52]:
curitiba_domicilio = curitiba_domicilio1[
    ["CD_BAIRRO", "NM_BAIRRO"] + variaveis_domicilio1
].merge(
    curitiba_domicilio2[
        ["CD_BAIRRO", "NM_BAIRRO"] + variaveis_domicilio2
    ],
    on=["CD_BAIRRO", "NM_BAIRRO"],
    how="inner",
    validate="1:1"
)

In [53]:
variaveis_saneamento = variaveis_domicilio1 + variaveis_domicilio2

In [54]:
saneamento_por_bairro = (
    curitiba_domicilio
    .groupby(["CD_BAIRRO", "NM_BAIRRO"], as_index=False)
    [variaveis_saneamento]
    .sum()
)

In [55]:
saneamento_por_bairro["domicilios_particulares_permanentemente_ocupados_2022"] = (
    saneamento_por_bairro["V00001"]
)

In [56]:
saneamento_por_bairro["sem_banheiro_sanitario_2022"] = (
    saneamento_por_bairro["V00238"]
)

In [57]:
saneamento_por_bairro["esgotamento_precario_2022"] = (
    saneamento_por_bairro["V00312"] +
    saneamento_por_bairro["V00313"] +
    saneamento_por_bairro["V00314"] +
    saneamento_por_bairro["V00316"]
)

In [58]:
saneamento_por_bairro["sem_rede_geral_agua_2022"] = (
    saneamento_por_bairro["V00464"]
)

In [59]:
saneamento_por_bairro["lixo_destino_inadequado_2022"] = (
    saneamento_por_bairro["V00399"] +
    saneamento_por_bairro["V00400"] +
    saneamento_por_bairro["V00401"] +
    saneamento_por_bairro["V00402"]
)

In [60]:
saneamento_por_bairro["domicilios_improvisados_estrutura_degradada_2022"] = (
    saneamento_por_bairro["V00002"] +
    saneamento_por_bairro["V00052"]
)

In [61]:
denominador_saneamento = (
    saneamento_por_bairro["domicilios_particulares_permanentemente_ocupados_2022"]
)

In [62]:
saneamento_por_bairro["pct_sem_banheiro_sanitario_2022"] = (
    saneamento_por_bairro["sem_banheiro_sanitario_2022"] /
    denominador_saneamento
    ) * 100

In [63]:
saneamento_por_bairro["pct_esgotamento_precario_2022"] = (
    saneamento_por_bairro["esgotamento_precario_2022"] /
    denominador_saneamento
    ) * 100

In [64]:
saneamento_por_bairro["pct_sem_rede_geral_agua_2022"] = (
    saneamento_por_bairro["sem_rede_geral_agua_2022"] /
    denominador_saneamento
    ) * 100

In [65]:
saneamento_por_bairro["pct_lixo_destino_inadequado_2022"] = (
    saneamento_por_bairro["lixo_destino_inadequado_2022"] /
    denominador_saneamento
    ) * 100

In [66]:
saneamento_por_bairro["pct_domicilios_improvisados_estrutura_degradada_2022"] = (
    saneamento_por_bairro["domicilios_improvisados_estrutura_degradada_2022"] /
    denominador_saneamento
    ) * 100

In [67]:
saneamento_bairros_final = saneamento_por_bairro[
    [
        "CD_BAIRRO",
        "NM_BAIRRO",
        "domicilios_particulares_permanentemente_ocupados_2022",
        "sem_banheiro_sanitario_2022",
        "pct_sem_banheiro_sanitario_2022",
        "esgotamento_precario_2022",
        "pct_esgotamento_precario_2022",
        "sem_rede_geral_agua_2022",
        "pct_sem_rede_geral_agua_2022",
        "lixo_destino_inadequado_2022",
        "pct_lixo_destino_inadequado_2022",
        "domicilios_improvisados_estrutura_degradada_2022",
        "pct_domicilios_improvisados_estrutura_degradada_2022"
    ]
].copy()

In [68]:
saneamento_bairros_final.head(3)

,CD_BAIRRO,NM_BAIRRO,domicilios_particulares_permanentemente_ocupados_2022,sem_banheiro_sanitario_2022,pct_sem_banheiro_sanitario_2022,esgotamento_precario_2022,pct_esgotamento_precario_2022,sem_rede_geral_agua_2022,pct_sem_rede_geral_agua_2022,lixo_destino_inadequado_2022,pct_lixo_destino_inadequado_2022,domicilios_improvisados_estrutura_degradada_2022,pct_domicilios_improvisados_estrutura_degradada_2022
0,4106902001,Centro,22215,0,0.0,0,0.0,0,0.0,26,0.117038,0,0.0
1,4106902002,São Francisco,2394,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,4106902003,Centro Cívico,2348,0,0.0,0,0.0,22,0.936968,0,0.0,0,0.0


# Unificando os dados

In [69]:
def limpar_texto(valor):
    if pd.isna(valor):
        return pd.NA
    
    valor = str(valor).strip().lower()
    
    valor = unicodedata.normalize("NFKD", valor)
    valor = valor.encode("ascii", "ignore").decode("utf-8")
    
    valor = re.sub(r"[^a-z0-9\s]", " ", valor)
    valor = re.sub(r"\s+", " ", valor).strip()
    
    return valor if valor else pd.NA

In [70]:
pop = populacao_por_bairro[
    [ 
        "CD_BAIRRO",
        "NM_BAIRRO",
        "populacao_2022"
    ]
].copy()

In [71]:
renda = curitiba_pessoa_renda[
    [
        "CD_BAIRRO",
        "NM_BAIRRO",
        "resp_domicilios_particulares",
        "moradores_domicilios_particulares",
        "rendimento_medio_responsavel",
        "rendimento_medio_responsavel_sm"
    ]
].copy()

In [72]:
colunas_renda_numericas = [
    "resp_domicilios_particulares",
    "moradores_domicilios_particulares",
    "rendimento_medio_responsavel",
    "rendimento_medio_responsavel_sm"
]

In [73]:
for coluna in colunas_renda_numericas:
    renda[coluna] = pd.to_numeric(renda[coluna], errors="coerce")

In [74]:
alf = curitiba_alfabetizacao[
    [
        "CD_BAIRRO",
        "NM_BAIRRO",
        "pop_15mais",
        "alfabetizados_15mais",
        "pct_alfabetizacao_15mais",
        "pct_analfabetismo_15mais"
    ]
].copy()

In [75]:
alf["analfabetos_15mais"] = (
    alf["pop_15mais"] - alf["alfabetizados_15mais"]
)

In [76]:
san = saneamento_bairros_final[
    [
        "CD_BAIRRO",
        "NM_BAIRRO",
        "domicilios_particulares_permanentemente_ocupados_2022",
        "sem_banheiro_sanitario_2022",
        "pct_sem_banheiro_sanitario_2022",
        "esgotamento_precario_2022",
        "pct_esgotamento_precario_2022",
        "sem_rede_geral_agua_2022",
        "pct_sem_rede_geral_agua_2022",
        "lixo_destino_inadequado_2022",
        "pct_lixo_destino_inadequado_2022",
        "domicilios_improvisados_estrutura_degradada_2022",
        "pct_domicilios_improvisados_estrutura_degradada_2022"
    ]
].copy()

In [77]:
base_bairros_2022 = (
    pop 
    .merge(
        renda, 
        on=["CD_BAIRRO", "NM_BAIRRO"], 
        how="outer",
        validate="1:1"
    )
    .merge(
        alf,
        on=["CD_BAIRRO", "NM_BAIRRO"], 
        how="outer", 
        validate="1:1",
    )
    .merge(
        san,
        on=["CD_BAIRRO", "NM_BAIRRO"],
        how="outer",
        validate="1:1",
    )
)

In [78]:
base_bairros_2022["NM_BAIRRO"] = base_bairros_2022["NM_BAIRRO"].apply(limpar_texto)

base_bairros_2022 = (
    base_bairros_2022 
    .drop(columns=["CD_BAIRRO"])
    .rename(columns={"NM_BAIRRO": "ATENDIMENTO_BAIRRO_NOME"})
)

base_bairros_2022 = base_bairros_2022.rename(
    columns={
        coluna: f"{coluna}_2022"
        for coluna in base_bairros_2022.columns 
        if coluna != "ATENDIMENTO_BAIRRO_NOME"
        and not coluna.endswith("_2022")
    }
)

base_bairros_2022 = (
    base_bairros_2022 
    .sort_values("ATENDIMENTO_BAIRRO_NOME")
    .reset_index(drop=True)
)

In [79]:
base_bairros_2022

,ATENDIMENTO_BAIRRO_NOME,populacao_2022,resp_domicilios_particulares_2022,moradores_domicilios_particulares_2022,rendimento_medio_responsavel_2022,rendimento_medio_responsavel_sm_2022,pop_15mais_2022,alfabetizados_15mais_2022,pct_alfabetizacao_15mais_2022,pct_analfabetismo_15mais_2022,...,sem_banheiro_sanitario_2022,pct_sem_banheiro_sanitario_2022,esgotamento_precario_2022,pct_esgotamento_precario_2022,sem_rede_geral_agua_2022,pct_sem_rede_geral_agua_2022,lixo_destino_inadequado_2022,pct_lixo_destino_inadequado_2022,domicilios_improvisados_estrutura_degradada_2022,pct_domicilios_improvisados_estrutura_degradada_2022
0,abranches,13991,5076,13944,5118.9,4.223515,11469,11268,98.24745,1.75255,...,0,0.0,36,0.70922,13,0.256107,7,0.137904,0,0.0
1,agua verde,49573,21971,49322,10406.93,8.586576,43803,43674,99.7055,0.2945,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
2,ahu,11859,4858,11771,11154.95,9.203754,10195,10175,99.803825,0.196175,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
3,alto boqueirao,51178,18440,51144,3221.02,2.657607,41970,41151,98.048606,1.951394,...,0,0.0,161,0.873102,18,0.097614,76,0.412148,0,0.0
4,alto da gloria,6242,3159,6242,10307.91,8.504876,5646,5638,99.858307,0.141693,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,uberaba,73241,26233,73104,4360.44,3.597723,59933,58782,98.079522,1.920478,...,5,0.01906,18,0.068616,386,1.471429,3,0.011436,9,0.034308
71,umbara,21969,7655,21955,3345.59,2.760388,17515,17117,97.727662,2.272338,...,0,0.0,805,10.516003,122,1.59373,0,0.0,0,0.0
72,vila izabel,12859,6071,12829,8305.04,6.852343,11395,11348,99.587538,0.412462,...,0,0.0,0,0.0,0,0.0,0,0.0,0,0.0
73,vista alegre,10790,4006,10724,8084.84,6.67066,9160,9099,99.334061,0.665939,...,0,0.0,19,0.474289,7,0.174738,0,0.0,0,0.0


In [80]:
base_bairros_2022["ATENDIMENTO_BAIRRO_NOME"].unique()

<StringArray>
[                    'abranches',                    'agua verde',
                           'ahu',                'alto boqueirao',
                'alto da gloria',                'alto da rua xv',
                         'atuba',                       'augusta',
                     'bacacheri',                   'bairro alto',
                   'barreirinha',                         'batel',
                    'bigorrilho',                     'boa vista',
                    'bom retiro',                     'boqueirao',
                  'botiatuvinha',                        'cabral',
                     'cachoeira',                        'cajuru',
           'campina do siqueira',                'campo comprido',
              'campo de santana',               'capao da imbuia',
                    'capao raso',                    'cascatinha',
                       'caximba',                        'centro',
                 'centro civico', 'cidade indust

In [81]:
ajustes_ibge2022 = {
    "botiatuvinha": "butiatuvinha",
    "cidade industrial de curitiba": "cidade industrial"
}

In [82]:
base_bairros_2022["ATENDIMENTO_BAIRRO_NOME"] = (
    base_bairros_2022["ATENDIMENTO_BAIRRO_NOME"]
    .apply(limpar_texto)
    .replace(MAPA_BAIRRO)
    .replace(ajustes_ibge2022)
)

In [83]:
output_path = data_dir / "cleaned/base_bairros_2022.csv"

In [84]:
base_bairros_2022.to_csv(output_path, index=False, encoding="latin1")